# Health data cleaning

This notebook performs the requested cleaning steps with pandas. Place `Kaggle_HealthData.csv` beside the notebook before running **Run All**. Missing values are reported without dropping rows or filling values. Patient Satisfaction Score is then removed for the reason documented below.

In [1]:
import pandas as pd
from pathlib import Path

# Look beside the notebook first, then in the uploaded-data folder.
source = Path("Kaggle_HealthData.csv")
if not source.exists():
    source = Path("../upload/Kaggle_HealthData.csv")

# Preserve literal categories such as "None" in Department Referral.
# Treat empty or whitespace-only cells as missing.
df = pd.read_csv(source, dtype={"Patient Id": "string"}, keep_default_na=False)
df = df.replace(r"^\s*$", pd.NA, regex=True)
print(f"Rows: {len(df):,}; Columns: {len(df.columns)}")
df.head()

Rows: 9,216; Columns: 11


,Patient Id,Patient Admission Date,Patient Admission Time,Merged,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,780-96-6113,9/9/2024,9:25:00 AM,W. Breede,Female,63,African American,None,Not Admission,5,32
1,714-35-6722,9/9/2024,4:42:00 PM,Y. Baldetti,Male,31,Asian,Orthopedics,Not Admission,<NA>,22
2,571-85-3714,9/9/2024,12:14:00 AM,M. Semerad,Male,75,White,General Practice,Not Admission,<NA>,16
3,404-43-9499,9/9/2024,8:33:00 PM,K. Blaydes,Male,79,African American,General Practice,Admission,<NA>,38
4,552-51-5855,9/9/2024,7:25:00 PM,F. Dickerson,Female,24,African American,None,Admission,<NA>,36


## 1. Convert the admission date

The source uses day/month/year. `pd.to_datetime` keeps the column as a datetime type rather than a text representation. Invalid nonblank dates raise an error so they are not silently lost.

In [2]:
df["Patient Admission Date"] = pd.to_datetime(
    df["Patient Admission Date"], format="%d/%m/%Y", errors="raise"
)
print("Date column type:", df["Patient Admission Date"].dtype)
df[["Patient Admission Date"]].head()

Date column type: datetime64[ns]


,Patient Admission Date
0,2024-09-09
1,2024-09-09
2,2024-09-09
3,2024-09-09
4,2024-09-09


## 2. Remove hyphens from patient IDs

Keep IDs as strings to preserve any leading zeros.

In [3]:
df["Patient Id"] = df["Patient Id"].str.replace("-", "", regex=False)
df[["Patient Id"]].head()

,Patient Id
0,780966113
1,714356722
2,571853714
3,404439499
4,552515855


## 3. Rename the merged column

In [4]:
df = df.rename(columns={"Merged": "pt name"})
df[["pt name"]].head()

,pt name
0,W. Breede
1,Y. Baldetti
2,M. Semerad
3,K. Blaydes
4,F. Dickerson


## 4. Count missing values for every column

Literal `None` in Department Referral means no referral and is retained as a category. Empty cells are counted as missing.

In [5]:
missing_values = df.isna().sum().rename("Missing values").to_frame()
missing_values.index.name = "Column"
missing_values

,Missing values
Column,
Patient Id,0
Patient Admission Date,0
Patient Admission Time,0
pt name,0
Patient Gender,0
Patient Age,0
Patient Race,0
Department Referral,0
Patient Admission Flag,0


## 5. Convert admission time

Parse the source AM/PM values and store Python `datetime.time` objects, displayed in 24-hour HH:MM:SS format. Pandas shows a time-only column as `object`; its nonmissing values are actual time objects.

In [6]:
from datetime import time

df["Patient Admission Time"] = pd.to_datetime(
    df["Patient Admission Time"], format="%I:%M:%S %p", errors="raise"
).dt.time

assert df["Patient Admission Time"].dropna().map(lambda value: isinstance(value, time)).all()
print("All nonmissing admission times are Python time objects.")
df[["Patient Admission Time"]].head()

All nonmissing admission times are Python time objects.


,Patient Admission Time
0,09:25:00
1,16:42:00
2,00:14:00
3,20:33:00
4,19:25:00


## 6. Drop Patient Satisfaction Score

Patient Satisfaction Score is not needed for this analysis and has more than 70% missing values: 6,699 of 9,216 records (72.69%). The missing-value report above retains the count for this column before removal.

In [7]:
missing_pct = df["Patient Satisfaction Score"].isna().mean() * 100
print(f"Patient Satisfaction Score missing: {missing_pct:.2f}%")
df = df.drop(columns=["Patient Satisfaction Score"])
print("Dropped Patient Satisfaction Score because it is not needed for this analysis and has >70% missing values.")
print(f"Remaining columns: {len(df.columns)}")

Patient Satisfaction Score missing: 72.69%
Dropped Patient Satisfaction Score because it is not needed for this analysis and has >70% missing values.
Remaining columns: 10


## Cleaned data preview

In [8]:
df.head()

,Patient Id,Patient Admission Date,Patient Admission Time,pt name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Waittime
0,780966113,2024-09-09,09:25:00,W. Breede,Female,63,African American,None,Not Admission,32
1,714356722,2024-09-09,16:42:00,Y. Baldetti,Male,31,Asian,Orthopedics,Not Admission,22
2,571853714,2024-09-09,00:14:00,M. Semerad,Male,75,White,General Practice,Not Admission,16
3,404439499,2024-09-09,20:33:00,K. Blaydes,Male,79,African American,General Practice,Admission,38
4,552515855,2024-09-09,19:25:00,F. Dickerson,Female,24,African American,None,Admission,36
